In [ ]:
import torch
import numpy as np
import random
from datasets import load_dataset

dataset = load_dataset('imdb', split = 'train')

np.random.seed(100)
idx = np.random.randint(len(dataset), size = 200)

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding


In [ ]:
from transformers import AutoTokenizer
from transformers import BertModel
from transformers import RobertaModel
from transformers import DistilBertModel

def get_model(model_name):
    checkpoint_names = {
        'bert': 'bert-base-cased',
        'roberta': 'roberta-base',
        'distilbert': 'distilbert-base-cased',
    }

    model_classes = {
        'bert': BertModel,
        'roberta': RobertaModel,
        'distilbert': DistilBertModel
    }
    return AutoTokenizer.from_pretrained(checkpoint_names[model_name]), model_classes[model_name].from_pretrained(checkpoint_names[model_name])

In [ ]:
from tqdm import tqdm

@torch.inference_mode()
def get_embedding_labels(model, loader):
    model.eval()

    total_embeddings = []
    labels = []

    for batch in tqdm(loader):
        labels.append(batch['labels'].unsqueeze(1))

        batch = {key: batch[key].to(device) for key in ['attention_mask', 'input_ids']}

        embeddings = model(**batch)['last_hidden_state'][:, 0, :]
        total_embeddings.append(embeddings.cpu())
    return torch.cat(total_embeddings, dim = 0), torch.cat(labels, dim = 0).to(torch.float32)

In [ ]:
def tokenization(example: str):
    return tokenizer(example['text'], add_special_tokens = True, return_token_type_ids = False, truncation = True)

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

### Через BERT

In [ ]:
tokenizer, model = get_model('bert')

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

In [ ]:

train_dataset = dataset.map(tokenization, batched = True)

In [ ]:
train_dataset = train_dataset.select(idx)

In [ ]:
train_dataset = train_dataset.remove_columns(['text'])
train_dataset = train_dataset.rename_column('label', 'labels')

In [ ]:
train_loader = DataLoader(train_dataset, batch_size = 32, collate_fn = data_collator, pin_memory = True, shuffle = False)

In [ ]:
model = model.to(device)

In [ ]:
train_embeddings = get_embedding_labels(model, train_loader)

In [ ]:
train_embeddings[0].shape

In [ ]:
torch.save(train_embeddings[0], 'train_embeddings.pt')

### Через Robert'у

In [ ]:
dataset = load_dataset("imdb", split="train")

np.random.seed(100)
idx = np.random.randint(len(dataset), size=200)

In [ ]:
tokenizer, model = get_model('roberta')

In [ ]:
train_dataset = dataset.map(tokenization, batched = True)

In [ ]:
train_dataset = train_dataset.select(idx)

In [ ]:
train_dataset = train_dataset.remove_columns(['text'])
train_dataset = train_dataset.rename_column('label', 'labels')

In [ ]:
model = model.to(device)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

train_loader = DataLoader(train_dataset, batch_size = 32, collate_fn = data_collator, pin_memory = True, shuffle = False)

In [ ]:
train_embeddings = get_embedding_labels(model, train_loader)

In [ ]:
train_embeddings[0].shape

In [ ]:
torch.save(train_embeddings[0], 'train_embeddings_roberta_2.pt')

### Через DistilBERT

In [ ]:
dataset = load_dataset("imdb", split="train")

np.random.seed(100)
idx = np.random.randint(len(dataset), size=200)

In [ ]:
tokenizer, model = get_model('distilbert')

In [ ]:
train_dataset = dataset.map(tokenization, batched = True)

In [ ]:
train_dataset = train_dataset.select(idx)

train_dataset = train_dataset.remove_columns(['text'])
train_dataset = train_dataset.rename_column('label', 'labels')


In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
model = model.to(device)

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

train_loader = DataLoader(train_dataset, batch_size = 32, collate_fn = data_collator, pin_memory = True, shuffle = False)

In [ ]:
train_embeddings = get_embedding_labels(model, train_loader)

In [ ]:
torch.save(train_embeddings[0], 'distilbert_embeddings.pt')